# Session 4 · Part 4 (optional) — Getting data from an API

So far the data came from files. A lot of financial data comes from **APIs**: a website you ask for data from code.
Here we use `yfinance`, a free (unofficial) way to get Yahoo Finance prices.

If 40 people hit Yahoo at once from the same building, it sometimes refuses with *"Too Many Requests"*. In that case the notebook automatically uses a saved file of Apple prices instead, so everything below still works.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

TICKER = "AAPL"          # try others: "MSFT", "NVDA", "EMAAR.AE"

try:
    import yfinance as yf
    prices = yf.download(TICKER, start="2024-01-01", end="2025-01-01", auto_adjust=True, progress=False)
    if isinstance(prices.columns, pd.MultiIndex):          # newer yfinance versions add a 'Ticker' level
        prices.columns = prices.columns.get_level_values(0)
    if prices.empty:
        raise ValueError("no data returned")
    source = f"Yahoo Finance ({TICKER})"
except Exception as e:
    print("Download didn't work:", type(e).__name__, "-> using the saved backup file")
    prices = pd.read_csv("data/aapl_prices_backup.csv", parse_dates=["Date"], index_col="Date")
    TICKER, source = "AAPL", "saved backup (AAPL, 2015-2017)"

print("Data from:", source, "|", len(prices), "trading days")
prices.head()

### Closing price over time

In [ ]:
prices["Close"].plot(figsize=(10, 4), title=f"{TICKER} closing price")
plt.ylabel("USD");

### Daily returns
The percentage change from one day's close to the next. `pct_change()` does it for the whole column in one go.

In [ ]:
prices["daily_return_pct"] = prices["Close"].pct_change() * 100
prices["daily_return_pct"].plot(figsize=(10, 4), title=f"{TICKER} daily return")
plt.ylabel("% change");

### 7-day rolling average
Smooths out the day-to-day noise.

In [ ]:
prices["rolling_7d"] = prices["Close"].rolling(window=7).mean()
prices[["Close", "rolling_7d"]].plot(figsize=(10, 4), title=f"{TICKER}: close vs 7-day average")
plt.ylabel("USD");

### Big moves
Which days moved more than 2% (up or down)?

In [ ]:
big_moves = prices[prices["daily_return_pct"].abs() > 2]
print(len(big_moves), "days with a move bigger than 2%")
big_moves[["Close", "daily_return_pct"]].round(2).head(10)

In [ ]:
prices.to_csv(f"data/{TICKER}_prices.csv")
print("Saved")